In [1]:
!pip -q install pyspark
import time
import os
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col, when, count, avg, sum, dense_rank, spark_partition_id, to_timestamp, year, month
from pyspark.sql.window import Window
from pyspark import StorageLevel

spark = SparkSession.builder.appName("EnergyAssessment").getOrCreate()

def load_csv(filename):
    return spark.read.csv(filename, header=True, inferSchema=True)

In [3]:
uploaded= files.upload()

Saving maintenance_tickets_1600.csv to maintenance_tickets_1600.csv
Saving meter_master.csv to meter_master.csv
Saving meter_readings_90000.csv to meter_readings_90000.csv
Saving meter_readings_extra_12000.csv to meter_readings_extra_12000.csv
Saving substation_master.csv to substation_master.csv
Saving tariff_master.csv to tariff_master.csv


**SECTION 01 / DATA PREPARATION & ANALYSIS** ->
**A. Inspect, select and transform**

In [4]:
# 01. Load all six CSVs. Display the schema, row count and first 10 rows of the main readings DataFrame.
readings_df = load_csv("meter_readings_90000.csv")
readings_extra_df = load_csv("meter_readings_extra_12000.csv")
meter_master_df = load_csv("meter_master.csv")
substation_master_df = load_csv("substation_master.csv")
tariff_master_df = load_csv("tariff_master.csv")
tickets_df = load_csv("maintenance_tickets_1600.csv")

readings_df.printSchema()
print(f"Total Rows: {readings_df.count()}")
readings_df.show(10)

root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

Total Rows: 90000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000001|   M0082|2026-0

In [5]:
# 02. Select reading_id, meter_id, reading_timestamp, energy_kwh and reading_status. In a separate DataFrame, rename source_type to collection_mode.
selected_df = readings_df.select("reading_id", "meter_id", "reading_timestamp", "energy_kwh", "reading_status")
selected_df.show(5)

renamed_df = readings_df.withColumnRenamed("source_type", "collection_mode")
renamed_df.select("reading_id", "collection_mode").show(5)

+----------+--------+-------------------+----------+--------------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|reading_status|
+----------+--------+-------------------+----------+--------------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|         Valid|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|         Valid|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|     Estimated|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|         Valid|
| RD0000005|   M0135|2026-05-27 01:45:00|      4.39|         Valid|
+----------+--------+-------------------+----------+--------------+
only showing top 5 rows
+----------+---------------+
|reading_id|collection_mode|
+----------+---------------+
| RD0000001|     SmartMeter|
| RD0000002|     SmartMeter|
| RD0000003|     SmartMeter|
| RD0000004|         Manual|
| RD0000005|     SmartMeter|
+----------+---------------+
only showing top 5 rows


In [6]:
# 03. Filter readings where reading_status is Valid and energy_kwh exceeds 20. Sort by energy consumed, highest first.
filtered_sorted_df = readings_df.filter(
    (col("reading_status") == "Valid") & (col("energy_kwh") > 20)
).orderBy(col("energy_kwh").desc())

filtered_sorted_df.show(10)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0041703|   M0101|2026-03-06 22:30:00|     110.0|    213.1|       0.822|             0|         Valid| SmartMeter|                 31.1|      OP010|
| RD0077494|   M0014|2026-05-05 17:30:00|    109.99|     NULL|       0.771|             0|         Valid|     Manual|                 39.0|      OP031|
| RD0039063|   M0101|2026-02-01 00:45:00|    109.96|    231.6|       0.835|             0|         Valid| SmartMeter|                 20.1|      OP021|
| RD0034412|   M0026|2026-02-24 21:15:00|    109.96|    231.6|       0.964|             

In [7]:
# 04. Create energy_band using when() / otherwise(): Low below 5 kWh, Medium from 5 through 25 kWh, High above 25 kWh, and Unknown for nulls.
banded_df = readings_df.withColumn(
    "energy_band",
    when(col("energy_kwh").isNull(), "Unknown")
    .when(col("energy_kwh") < 5, "Low")
    .when((col("energy_kwh") >= 5) & (col("energy_kwh") <= 25), "Medium")
    .when(col("energy_kwh") > 25, "High")
    .otherwise("Unknown")
)

banded_df.select("reading_id", "energy_kwh", "energy_band").show(10)

+----------+----------+-----------+
|reading_id|energy_kwh|energy_band|
+----------+----------+-----------+
| RD0000001|     22.53|     Medium|
| RD0000002|      2.97|        Low|
| RD0000003|      8.61|     Medium|
| RD0000004|      1.87|        Low|
| RD0000005|      4.39|        Low|
| RD0000006|      0.36|        Low|
| RD0000007|     79.47|       High|
| RD0000008|     28.35|       High|
| RD0000009|      1.96|        Low|
| RD0000010|     22.52|     Medium|
+----------+----------+-----------+
only showing top 10 rows


**B. Null values and data cleaning**

In [8]:
# 05. Calculate null counts for energy_kwh, voltage_v, power_factor and operator_id.
null_counts_df = readings_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["energy_kwh", "voltage_v", "power_factor", "operator_id"]
])

null_counts_df.show()

+----------+---------+------------+-----------+
|energy_kwh|voltage_v|power_factor|operator_id|
+----------+---------+------------+-----------+
|      1620|     2070|        1440|       5400|
+----------+---------+------------+-----------+



In [9]:
# 06. Create a new DataFrame that fills missing operator_id with UNASSIGNED and missing power_factor with 0.90. Do not modify the original DataFrame.
filled_df = readings_df.na.fill({
    "operator_id": "UNASSIGNED",
    "power_factor": 0.90
})

filled_df.select("reading_id", "operator_id", "power_factor").show(10)

+----------+-----------+------------+
|reading_id|operator_id|power_factor|
+----------+-----------+------------+
| RD0000001|      OP029|       0.939|
| RD0000002|      OP025|       0.845|
| RD0000003|      OP007|       0.879|
| RD0000004| UNASSIGNED|       0.831|
| RD0000005|      OP019|       0.966|
| RD0000006|      OP021|       0.788|
| RD0000007|      OP016|       0.986|
| RD0000008|      OP025|       0.818|
| RD0000009|      OP010|       0.817|
| RD0000010|      OP028|       0.804|
+----------+-----------+------------+
only showing top 10 rows


In [10]:
# 07. Remove readings with missing energy_kwh or voltage_v. Report the number of rows removed.
initial_count = readings_df.count()
cleaned_readings_df = readings_df.dropna(subset=["energy_kwh", "voltage_v"])
final_count = cleaned_readings_df.count()

print(f"Initial row count: {initial_count}")
print(f"Cleaned row count: {final_count}")
print(f"Rows removed: {initial_count - final_count}")

Initial row count: 90000
Cleaned row count: 86349
Rows removed: 3651


**C. Joins, groupBy and aggregations**

In [11]:
# 08. Join readings with meter_master on meter_id. Calculate total energy for Valid readings by site_type.
total_energy_site = readings_df.filter(col("reading_status") == "Valid") \
    .join(meter_master_df, on="meter_id", how="inner") \
    .groupBy("site_type") \
    .agg(sum("energy_kwh").alias("total_valid_energy_kwh"))

total_energy_site.show()

+---------------+----------------------+
|      site_type|total_valid_energy_kwh|
+---------------+----------------------+
|Public Facility|             138870.41|
|    Residential|     100863.2900000004|
|     Industrial|     710317.4100000021|
|     Commercial|     451525.1799999996|
+---------------+----------------------+



In [12]:
# 09. Extend the join to substation_master. Calculate total Valid energy by region and city.
total_energy_region_city = readings_df.filter(col("reading_status") == "Valid") \
    .join(meter_master_df, on="meter_id", how="inner") \
    .join(substation_master_df, on="substation_id", how="inner") \
    .groupBy("region", "city") \
    .agg(sum("energy_kwh").alias("total_valid_energy_kwh"))

total_energy_region_city.show()

+-------+-------------+----------------------+
| region|         city|total_valid_energy_kwh|
+-------+-------------+----------------------+
|Central|       Indore|    176503.91000000012|
|   West|       Mumbai|     52116.20999999999|
|  South|    Hyderabad|    51937.789999999884|
|  South|Visakhapatnam|    205015.82999999932|
|   West|    Ahmedabad|     138627.7400000004|
|  North|      Lucknow|     142610.3999999998|
|  South|    Bengaluru|     73787.59999999998|
|  North|       Jaipur|     47863.15999999996|
|  South|        Kochi|     204272.4600000002|
|  North|        Delhi|     94234.89000000014|
|  South|      Chennai|     51829.05999999996|
|   West|         Pune|    162777.23999999996|
+-------+-------------+----------------------+



In [13]:
# 10. Join tariff_master through meter_master. For non-Rejected readings with non-null energy, calculate energy_charge_inr = energy_kwh * rate_per_kwh_inr. Do not add the monthly fixed charge to every reading.
energy_charges_df = readings_df.filter(
    (col("reading_status") != "Rejected") & (col("energy_kwh").isNotNull())
).join(meter_master_df, on="meter_id", how="inner") \
 .join(tariff_master_df, on="tariff_id", how="inner") \
 .withColumn("energy_charge_inr", col("energy_kwh") * col("rate_per_kwh_inr"))

energy_charges_df.select("reading_id", "meter_id", "tariff_id", "energy_kwh", "rate_per_kwh_inr", "energy_charge_inr").show(10)

+----------+--------+-------------------+----------+----------------+------------------+
|reading_id|meter_id|          tariff_id|energy_kwh|rate_per_kwh_inr| energy_charge_inr|
+----------+--------+-------------------+----------+----------------+------------------+
| RD0000001|   M0082|2026-10-30 05:00:00|     22.53|             6.7|150.95100000000002|
| RD0000002|   M0020|2026-10-30 02:00:00|      2.97|             6.4|19.008000000000003|
| RD0000003|   M0054|2026-10-30 03:00:00|      8.61|             8.2| 70.60199999999999|
| RD0000004|   M0111|2026-10-30 01:00:00|      1.87|             5.1| 9.536999999999999|
| RD0000005|   M0135|2026-10-30 03:00:00|      4.39|             8.2|            35.998|
| RD0000006|   M0139|2026-10-30 02:00:00|      0.36|             6.4|             2.304|
| RD0000007|   M0142|2026-10-30 04:00:00|     79.47|             9.1| 723.1769999999999|
| RD0000009|   M0005|2026-10-30 01:00:00|      1.96|             5.1| 9.995999999999999|
| RD0000010|   M0052|

In [14]:
# 11. Join maintenance tickets with meter_master. Find ticket counts and average maintenance_cost_inr by site_type and severity.
tickets_summary_df = tickets_df.join(meter_master_df, on="meter_id", how="inner") \
    .groupBy("site_type", "severity") \
    .agg(
        count("ticket_id").alias("ticket_count"),
        avg("maintenance_cost_inr").alias("avg_maintenance_cost_inr")
    ) \
    .orderBy("site_type", "severity")

tickets_summary_df.show()

+---------------+--------+------------+------------------------+
|      site_type|severity|ticket_count|avg_maintenance_cost_inr|
+---------------+--------+------------+------------------------+
|     Commercial|Critical|          37|      13424.726216216219|
|     Commercial|    High|         130|      12293.073793103451|
|     Commercial|     Low|         166|      12223.517692307689|
|     Commercial|  Medium|         171|      13706.600886075952|
|     Industrial|Critical|          17|      12296.198000000002|
|     Industrial|    High|          63|      12838.869344262299|
|     Industrial|     Low|          66|       12717.52508474576|
|     Industrial|  Medium|         106|      12043.898526315788|
|Public Facility|Critical|          15|       8986.843571428572|
|Public Facility|    High|          48|      12905.279347826086|
|Public Facility|     Low|          60|      11195.929464285715|
|Public Facility|  Medium|          97|      13050.628421052634|
|    Residential|Critical

In [15]:
# 12. Calculate average voltage_v and total outage_minutes by substation. Show the five substations with the highest total outage minutes.
substation_outage_df = readings_df.join(meter_master_df, on="meter_id", how="inner") \
    .groupBy("substation_id") \
    .agg(
        avg("voltage_v").alias("avg_voltage_v"),
        sum("outage_minutes").alias("total_outage_minutes")
    ) \
    .orderBy(col("total_outage_minutes").desc())

substation_outage_df.show(5)

+-------------+------------------+--------------------+
|substation_id|     avg_voltage_v|total_outage_minutes|
+-------------+------------------+--------------------+
|        SUB10|230.04010567581344|               91224|
|        SUB09|230.04586272640606|               81734|
|        SUB04| 230.0558687466385|               75872|
|        SUB08|229.88012557077673|               75581|
|        SUB03|230.03094896227708|               61662|
+-------------+------------------+--------------------+
only showing top 5 rows


In [16]:
# 13. Calculate the number of Valid, Estimated and Rejected readings for each source_type, ordered by reading count.
source_status_df = readings_df.groupBy("source_type") \
    .pivot("reading_status", ["Valid", "Estimated", "Rejected"]) \
    .agg(count("reading_id")) \
    .withColumn("total_readings", col("Valid") + col("Estimated") + col("Rejected")) \
    .orderBy(col("total_readings").desc())

source_status_df.show()

+-----------+-----+---------+--------+--------------+
|source_type|Valid|Estimated|Rejected|total_readings|
+-----------+-----+---------+--------+--------------+
| SmartMeter|65521|    10443|    3202|         79166|
|     Manual| 8965|     1405|     464|         10834|
+-----------+-----+---------+--------+--------------+



**SECTION 02 / ADVANCED DATAFRAME OPERATIONS**

**D. Date and window functions**

In [17]:
# 14. Convert reading_timestamp into a Spark timestamp. Extract the year and month and calculate monthly Valid energy consumption by region.
monthly_energy_df = readings_df.withColumn("timestamp", to_timestamp(col("reading_timestamp"))) \
    .filter(col("reading_status") == "Valid") \
    .withColumn("year", year("timestamp")) \
    .withColumn("month", month("timestamp")) \
    .join(meter_master_df, on="meter_id", how="inner") \
    .join(substation_master_df, on="substation_id", how="inner") \
    .groupBy("region", "year", "month") \
    .agg(sum("energy_kwh").alias("total_valid_energy_kwh")) \
    .orderBy("region", "year", "month")

monthly_energy_df.show()

+-------+----+-----+----------------------+
| region|year|month|total_valid_energy_kwh|
+-------+----+-----+----------------------+
|Central|2026|    1|     31475.57999999999|
|Central|2026|    2|    28161.410000000007|
|Central|2026|    3|              29805.85|
|Central|2026|    4|    29475.339999999982|
|Central|2026|    5|    30048.879999999986|
|Central|2026|    6|     27536.84999999999|
|  North|2026|    1|    48849.309999999925|
|  North|2026|    2|    45444.249999999985|
|  North|2026|    3|     51094.61000000006|
|  North|2026|    4|     43451.02000000005|
|  North|2026|    5|     48796.93000000004|
|  North|2026|    6|     47072.32999999992|
|  South|2026|    1|     97547.38999999987|
|  South|2026|    2|     90864.80000000006|
|  South|2026|    3|    102258.77000000005|
|  South|2026|    4|     97266.94000000005|
|  South|2026|    5|     101204.8699999999|
|  South|2026|    6|     97699.97000000002|
|   West|2026|    1|     59967.56999999997|
|   West|2026|    2|     55807.8

In [18]:
# 15. Find the top three meters by total Valid energy within each substation using dense_rank().
meter_substation_energy = readings_df.filter(col("reading_status") == "Valid") \
    .join(meter_master_df, on="meter_id", how="inner") \
    .groupBy("substation_id", "meter_id") \
    .agg(sum("energy_kwh").alias("total_energy_kwh"))

window_substation = Window.partitionBy("substation_id").orderBy(col("total_energy_kwh").desc())

top_meters_df = meter_substation_energy.withColumn("rank", dense_rank().over(window_substation)) \
    .filter(col("rank") <= 3)

top_meters_df.show(20)

+-------------+--------+------------------+----+
|substation_id|meter_id|  total_energy_kwh|rank|
+-------------+--------+------------------+----+
|        SUB01|   M0130|           9364.98|   1|
|        SUB01|   M0110| 9259.750000000002|   2|
|        SUB01|   M0068| 8832.580000000005|   3|
|        SUB02|   M0071|          29242.04|   1|
|        SUB02|   M0145|28641.319999999992|   2|
|        SUB02|   M0133| 7060.140000000003|   3|
|        SUB03|   M0096| 9632.740000000003|   1|
|        SUB03|   M0080| 9525.940000000002|   2|
|        SUB03|   M0073| 9369.789999999995|   3|
|        SUB04|   M0038|          29505.88|   1|
|        SUB04|   M0099|29216.319999999996|   2|
|        SUB04|   M0098|28613.460000000014|   3|
|        SUB05|   M0094| 9659.639999999996|   1|
|        SUB05|   M0050| 9570.859999999997|   2|
|        SUB05|   M0008| 9524.780000000004|   3|
|        SUB06|   M0126|32573.380000000023|   1|
|        SUB06|   M0128|30748.840000000004|   2|
|        SUB06|   M0

In [19]:
# 16. Calculate cumulative Valid energy for each meter using a window ordered by reading timestamp and reading ID.
cum_window = Window.partitionBy("meter_id") \
    .orderBy(col("reading_timestamp"), col("reading_id")) \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

cumulative_energy_df = readings_df.filter(col("reading_status") == "Valid") \
    .withColumn("cumulative_energy_kwh", sum("energy_kwh").over(cum_window))

cumulative_energy_df.select("meter_id", "reading_id", "reading_timestamp", "energy_kwh", "cumulative_energy_kwh").show(15)

+--------+----------+-------------------+----------+---------------------+
|meter_id|reading_id|  reading_timestamp|energy_kwh|cumulative_energy_kwh|
+--------+----------+-------------------+----------+---------------------+
|   M0001| RD0083587|2026-01-01 01:15:00|     22.27|                22.27|
|   M0001| RD0086275|2026-01-01 03:30:00|      4.05|                26.32|
|   M0001| RD0088204|2026-01-01 09:45:00|     11.28|                 37.6|
|   M0001| RD0005149|2026-01-02 04:45:00|     11.76|                49.36|
|   M0001| RD0035530|2026-01-02 08:15:00|     10.06|                59.42|
|   M0001| RD0070104|2026-01-02 18:30:00|     15.83|                75.25|
|   M0001| RD0006677|2026-01-03 00:30:00|     17.06|                92.31|
|   M0001| RD0021042|2026-01-03 06:00:00|     17.14|               109.45|
|   M0001| RD0032205|2026-01-03 08:30:00|     20.46|               129.91|
|   M0001| RD0074606|2026-01-03 18:15:00|      20.4|               150.31|
|   M0001| RD0019547|2026

**E. Union and deduplication**

In [20]:
# 17. Combine both readings files using unionByName(). Compare row counts before and after the union.
count_main = readings_df.count()
count_extra = readings_extra_df.count()

combined_readings_df = readings_df.unionByName(readings_extra_df)
count_combined = combined_readings_df.count()

print(f"Main readings count: {count_main}")
print(f"Extra readings count: {count_extra}")
print(f"Combined total count: {count_combined}")

Main readings count: 90000
Extra readings count: 12000
Combined total count: 102000


In [21]:
# 18. Identify repeated reading_id values, remove duplicates using that key, and report the final unique reading count.
duplicate_keys = combined_readings_df.groupBy("reading_id") \
    .agg(count("reading_id").alias("occurrence_count")) \
    .filter(col("occurrence_count") > 1)

print(f"Number of repeated reading_id keys: {duplicate_keys.count()}")

deduped_readings_df = combined_readings_df.dropDuplicates(["reading_id"])
print(f"Final unique reading count: {deduped_readings_df.count()}")

Number of repeated reading_id keys: 700
Final unique reading count: 101300


**F. Lazy execution, caching and persistence**

In [22]:
# 19. Build a pipeline with filter(), withColumn(), groupBy() and orderBy(). Inspect it using explain() before calling an action.
pipeline_df = readings_df.filter(col("reading_status") == "Valid") \
    .withColumn("energy_mwh", col("energy_kwh") / 1000) \
    .groupBy("source_type") \
    .agg(avg("energy_mwh").alias("avg_energy_mwh")) \
    .orderBy("avg_energy_mwh")

pipeline_df.explain(True)

== Parsed Logical Plan ==
'Sort ['avg_energy_mwh ASC NULLS FIRST], true
+- Aggregate [source_type#25], [source_type#25, avg(energy_mwh#990) AS avg_energy_mwh#991]
   +- Project [reading_id#17, meter_id#18, reading_timestamp#19, energy_kwh#20, voltage_v#21, power_factor#22, outage_minutes#23, reading_status#24, source_type#25, ambient_temperature_c#26, operator_id#27, (energy_kwh#20 / cast(1000 as double)) AS energy_mwh#990]
      +- Filter (reading_status#24 = Valid)
         +- Relation [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] csv

== Analyzed Logical Plan ==
source_type: string, avg_energy_mwh: double
Sort [avg_energy_mwh#991 ASC NULLS FIRST], true
+- Aggregate [source_type#25], [source_type#25, avg(energy_mwh#990) AS avg_energy_mwh#991]
   +- Project [reading_id#17, meter_id#18, reading_timestamp#19, energy_kwh#20, voltage_v#21, power_factor#2

In [23]:
# 20. Cache Valid readings, run count() twice, and record the elapsed time for each action. Check is_cached.
valid_readings_df = readings_df.filter(col("reading_status") == "Valid").cache()

print(f"Is cached: {valid_readings_df.is_cached}")

start_1 = time.time()
count_1 = valid_readings_df.count()
time_1 = time.time() - start_1
print(f"Run 1 count: {count_1} | Time taken: {time_1:.4f} seconds")

start_2 = time.time()
count_2 = valid_readings_df.count()
time_2 = time.time() - start_2
print(f"Run 2 count: {count_2} | Time taken: {time_2:.4f} seconds")

valid_readings_df.unpersist()

Is cached: True
Run 1 count: 74486 | Time taken: 4.1208 seconds
Run 2 count: 74486 | Time taken: 0.3003 seconds


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [24]:
# 21. Persist readings with outage_minutes > 0 using MEMORY_AND_DISK. Execute two actions and then release the data using unpersist().
outage_readings_df = readings_df.filter(col("outage_minutes") > 0).persist(StorageLevel.MEMORY_AND_DISK)

outage_count = outage_readings_df.count()
print(f"Action 1 (count): {outage_count}")

outage_readings_df.show(5)

outage_readings_df.unpersist()
print(f"Released cache. Is cached: {outage_readings_df.is_cached}")

Action 1 (count): 7653
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000050|   M0063|2026-01-12 18:30:00|     18.53|    241.5|       0.908|            11|         Valid| SmartMeter|                 32.8|       NULL|
| RD0000055|   M0031|2026-05-04 21:30:00|     15.21|    226.2|       0.895|           108|         Valid| SmartMeter|                 38.9|      OP037|
| RD0000085|   M0069|2026-02-22 19:15:00|     27.29|    221.1|       0.919|           172|         Valid| SmartMeter|                 31.1|      OP031|
| RD0000090|   M0121|2026-03-13 00:45:00|     35.94|    217.1|   

In [25]:
# 22. Compare repeated execution of the same aggregation with and without caching. Record observations; do not assume caching must be faster.
agg_query = readings_df.groupBy("reading_status").agg(avg("energy_kwh"), sum("voltage_v"))

t0 = time.time()
agg_query.collect()
time_uncached_1 = time.time() - t0

t0 = time.time()
agg_query.collect()
time_uncached_2 = time.time() - t0

cached_agg = agg_query.cache()
t0 = time.time()
cached_agg.collect()
time_cached_1 = time.time() - t0

t0 = time.time()
cached_agg.collect()
time_cached_2 = time.time() - t0

print(f"Uncached Run 1: {time_uncached_1:.4f}s | Uncached Run 2: {time_uncached_2:.4f}s")
print(f"Cached Run 1: {time_cached_1:.4f}s | Cached Run 2: {time_cached_2:.4f}s")

cached_agg.unpersist()

Uncached Run 1: 1.7176s | Uncached Run 2: 0.1166s
Cached Run 1: 0.1221s | Cached Run 2: 0.2726s


DataFrame[reading_status: string, avg(energy_kwh): double, sum(voltage_v): double]

**G. Partitions and output**

In [26]:
# 23. Repartition the main readings DataFrame into eight partitions and show the number of rows in each using spark_partition_id().
repartitioned_8_df = readings_df.repartition(8)

rows_per_partition = repartitioned_8_df.groupBy(spark_partition_id().alias("partition_id")) \
    .agg(count("*").alias("row_count")) \
    .orderBy("partition_id")

rows_per_partition.show()

+------------+---------+
|partition_id|row_count|
+------------+---------+
|           0|    11249|
|           1|    11249|
|           2|    11251|
|           3|    11251|
|           4|    11250|
|           5|    11250|
|           6|    11250|
|           7|    11250|
+------------+---------+



In [27]:
# 24. Start from 12 partitions. Reduce to three using coalesce(3) and separately using repartition(3). Compare the execution plans.
base_12_df = readings_df.repartition(12)

coalesced_3_df = base_12_df.coalesce(3)
repartitioned_3_df = base_12_df.repartition(3)

print("Execution Plan for coalesce(3)")
coalesced_3_df.explain()

print("\nExecution Plan for repartition(3)")
repartitioned_3_df.explain()

Execution Plan for coalesce(3)
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=2608]
      +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<reading_id:string,meter_id:string,reading_timestamp:timestamp,energy_kwh:double,voltage_v:...



Execution Plan for repartition(3)
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=2617]
   +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_te

In [28]:
# 25. Join cleaned readings with meter and substation details, then save the enriched DataFrame as Parquet using four output partitions. Inspect the resulting part files.
enriched_df = cleaned_readings_df.join(meter_master_df, on="meter_id", how="inner") \
    .join(substation_master_df, on="substation_id", how="inner")

output_path = "enriched_readings_parquet"
enriched_df.repartition(4).write.mode("overwrite").parquet(output_path)

print("Saved Parquet part files:")
for item in os.listdir(output_path):
    if item.endswith(".parquet") or item.startswith("part-"):
        print(item)

Saved Parquet part files:
part-00001-8b21804e-acb6-4726-8a26-c9c746486fff-c000.snappy.parquet
part-00003-8b21804e-acb6-4726-8a26-c9c746486fff-c000.snappy.parquet
part-00000-8b21804e-acb6-4726-8a26-c9c746486fff-c000.snappy.parquet
part-00002-8b21804e-acb6-4726-8a26-c9c746486fff-c000.snappy.parquet


**SECTION 03 / INTEGRATED ASSESSMENT**

In [29]:
# Final Capstone: Energy Consumption & Reliability Report

# 01 Combine the two readings datasets and deduplicate by reading_id.
combined_raw = readings_df.unionByName(readings_extra_df)
deduped_readings = combined_raw.dropDuplicates(["reading_id"])

# 02 Apply a documented cleaning rule for null energy values. Exclude Rejected readings from billable consumption.
billable_readings = deduped_readings.filter(
    (col("energy_kwh").isNotNull()) & (col("reading_status") != "Rejected")
)

# 03 Join meter, substation and tariff information using the appropriate keys.
full_joined_df = billable_readings.join(meter_master_df, on="meter_id", how="inner") \
    .join(substation_master_df, on="substation_id", how="inner") \
    .join(tariff_master_df, on="tariff_id", how="inner")

# 04 Create a month column from reading_timestamp and calculate total kWh and estimated energy charges.
# 05 Calculate total outage minutes and the percentage of readings that report an outage for each region and site type.
report_base = full_joined_df.withColumn("month", month(to_timestamp(col("reading_timestamp")))) \
    .withColumn("energy_charge_inr", col("energy_kwh") * col("rate_per_kwh_inr")) \
    .withColumn("has_outage", when(col("outage_minutes") > 0, 1).otherwise(0)) \
    .groupBy("region", "site_type", "month") \
    .agg(
        sum("energy_kwh").alias("total_kwh"),
        sum("energy_charge_inr").alias("total_energy_charges_inr"),
        sum("outage_minutes").alias("total_outage_minutes"),
        (sum("has_outage") / count("*") * 100).alias("outage_percentage")
    )

# 06 Rank site types within each region by total billable energy charges.
window_region = Window.partitionBy("region", "month").orderBy(col("total_energy_charges_inr").desc())
ranked_report_df = report_base.withColumn("rank", dense_rank().over(window_region))

# 07 Cache or persist the final aggregated report, run two actions, and release the cached data.
ranked_report_df.persist(StorageLevel.MEMORY_AND_DISK)

print(f"Report total aggregated rows: {ranked_report_df.count()}")
ranked_report_df.show(20, truncate=False)

ranked_report_df.unpersist()

# 08 Write the report as Parquet with two output partitions.
report_output_path = "final_energy_report_parquet"
ranked_report_df.coalesce(2).write.mode("overwrite").parquet(report_output_path)

print("Generated Parquet partitions:")
for file_name in os.listdir(report_output_path):
    if file_name.endswith(".parquet") or file_name.startswith("part-"):
        print(file_name)

Report total aggregated rows: 120
+------+---------------+-----+------------------+------------------------+--------------------+------------------+----+
|region|site_type      |month|total_kwh         |total_energy_charges_inr|total_outage_minutes|outage_percentage |rank|
+------+---------------+-----+------------------+------------------------+--------------------+------------------+----+
|North |Industrial     |7    |10280.45          |93552.09499999997       |585                 |3.614457831325301 |1   |
|North |Commercial     |7    |6708.58           |55010.355999999985      |2083                |7.4074074074074066|2   |
|North |Public Facility|7    |1963.1500000000003|13153.105               |604                 |4.827586206896552 |3   |
|North |Residential    |7    |819.8             |5050.406999999998       |864                 |4.5662100456621   |4   |
|South |Industrial     |5    |53718.419999999984|488837.622              |7731                |9.468822170900692 |1   |
|South